# 03 — Citation Grounding and Hallucination Check

Implements Chapter 3's post-generation `verify_citations` function: given a narrative and the source data it was supposedly grounded in, flags any citation that doesn't resolve — a **hallucinated citation**. Demonstrates both a clean narrative (all citations resolve) and a narrative with an injected hallucinated citation (flagged before it would reach a reviewer).

In [1]:
from dataclasses import dataclass, field

@dataclass
class Citation:
    source_type: str
    source_id: str

@dataclass
class CitedClaim:
    text: str
    citations: list = field(default_factory=list)

class SourceBundle:
    """Everything actually retrieved/queried for one generation call -- the ground truth
    a citation must resolve against."""
    def __init__(self, kyc: dict, transactions: list, case_notes: list):
        self.kyc_fields = set(kyc.keys())
        self.transaction_ids = {t['transaction_id'] for t in transactions}
        self.case_ids = {c['case_id'] for c in case_notes}

    def resolves(self, citation: Citation) -> bool:
        if citation.source_type == 'kyc_field':
            return citation.source_id in self.kyc_fields
        if citation.source_type == 'transaction_id':
            return citation.source_id in self.transaction_ids
        if citation.source_type == 'prior_case_id':
            return citation.source_id in self.case_ids
        return False

def verify_citations(all_claims: list, source_data: SourceBundle) -> list:
    """Returns citation strings that don't resolve against the actual retrieved/queried
    source data -- i.e. hallucinated citations."""
    broken = []
    for claim in all_claims:
        for citation in claim.citations:
            if not source_data.resolves(citation):
                broken.append(f"{citation.source_type}:{citation.source_id}")
    return broken

print('verify_citations() defined.')

verify_citations() defined.


In [2]:
source = SourceBundle(
    kyc={'occupation': 'x', 'risk_rating': 'x', 'stated_income': 'x'},
    transactions=[{'transaction_id': 'TXN-88201'}, {'transaction_id': 'TXN-88213'}, {'transaction_id': 'TXN-88240'}],
    case_notes=[{'case_id': 'CASE-2025-0033'}],
)

clean_claims = [
    CitedClaim('Consultant with MEDIUM risk rating.', [Citation('kyc_field', 'occupation'), Citation('kyc_field', 'risk_rating')]),
    CitedClaim('Three near-threshold transactions.', [Citation('transaction_id', 'TXN-88201'), Citation('transaction_id', 'TXN-88213'), Citation('transaction_id', 'TXN-88240')]),
    CitedClaim('Similar pattern previously closed as false positive.', [Citation('prior_case_id', 'CASE-2025-0033')]),
]

broken = verify_citations(clean_claims, source)
print('Clean narrative -- broken citations:', broken)
assert broken == [], 'A clean narrative should have zero unresolved citations.'
print('PASS: clean narrative has zero hallucinated citations.')

Clean narrative -- broken citations: []
PASS: clean narrative has zero hallucinated citations.


## Injecting a hallucinated citation

The same claim structure, but one claim now cites a transaction ID that was never actually retrieved for this generation call — exactly the kind of failure Chapter 6's bug narrative describes.

In [3]:
hallucinated_claims = clean_claims + [
    CitedClaim('An additional large wire transfer was also observed.', [Citation('transaction_id', 'TXN-99999')]),
    CitedClaim('Customer has a prior SAR filing on record.', [Citation('prior_case_id', 'CASE-2019-0001')]),
]

broken = verify_citations(hallucinated_claims, source)
print('Narrative with injected hallucinations -- broken citations:', broken)
assert len(broken) == 2, 'Both injected hallucinated citations must be caught.'
assert 'transaction_id:TXN-99999' in broken
assert 'prior_case_id:CASE-2019-0001' in broken
print('PASS: both hallucinated citations were caught before this narrative would reach a reviewer.')
print()
print('Per Chapter 3: this narrative would now be routed to review with these specific')
print('claims flagged, OR regenerated with the offending claims removed -- the conservative')
print('default is to flag rather than silently drop, so the reviewer sees exactly what to scrutinize.')

Narrative with injected hallucinations -- broken citations: ['transaction_id:TXN-99999', 'prior_case_id:CASE-2019-0001']
PASS: both hallucinated citations were caught before this narrative would reach a reviewer.

Per Chapter 3: this narrative would now be routed to review with these specific
claims flagged, OR regenerated with the offending claims removed -- the conservative
default is to flag rather than silently drop, so the reviewer sees exactly what to scrutinize.


## What this check does NOT catch

Chapter 3 is explicit about the limits of this check: it catches citations that don't resolve to anything real. It does **not** catch a citation that resolves correctly but is misinterpreted or overstated — that's still the human reviewer's job (Chapter 4).

In [4]:
# A citation that resolves fine (TXN-88201 IS real) but the claim built on it overstates it --
# this check cannot and should not try to catch this; it's a semantic-accuracy problem,
# not a grounding problem, and Chapter 3 is explicit that these are complementary controls.
overstated_claim = CitedClaim(
    'Customer has an extensive history of suspicious international wire fraud.',
    [Citation('transaction_id', 'TXN-88201')],
)
broken = verify_citations([overstated_claim], source)
print('Broken citations for the overstated claim:', broken, '(empty -- the citation IS real)')
print()
print('This is exactly the gap human review (Chapter 4) exists to close: the citation')
print('resolves, so this mechanical check passes -- but whether the CLAIM built on that')
print('citation is a fair characterization is a judgment call, not a grounding check.')

Broken citations for the overstated claim: [] (empty -- the citation IS real)

This is exactly the gap human review (Chapter 4) exists to close: the citation
resolves, so this mechanical check passes -- but whether the CLAIM built on that
citation is a fair characterization is a judgment call, not a grounding check.
